In [1]:
from stanbkt import load_model
from stanbkt.models import MultiBKT
from typing import cast

/home/sppradhan/Research/StanBKT-Experiments/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd

data = pd.read_csv("./data/itv.csv")

In [3]:
itv_model: MultiBKT = cast(MultiBKT, load_model("output/02_fit/itv.stanbktmod"))

In [4]:
itv_model.fits.get_fitted_kcs()

{'order-of-operations rule'}

In [5]:
fit = itv_model.fits.get_fit("order-of-operations rule")

In [6]:
from stanbkt.utils.data_utils import iter_kc_data
from stanbkt import ColumnNames

# extract the correct stan index to group from the data
# in newer stanbkt, this is stored in the fit object
column_names = {
    ColumnNames.PROBLEM_ID: "problem_id",
    ColumnNames.CORRECTNESS: "correct",
    ColumnNames.STUDENT_ID: "user_id",
    ColumnNames.KC_ID: "skill_name",
    ColumnNames.ORDER: "problem_id",
    ColumnNames.GROUP: "template_id",
}
kc_data = next(iter_kc_data(data, column_names, return_groups=True))
group_2_index = kc_data[1].group_2_index

In [7]:
def extract_draws(fit, param_name):
    draws_df = fit.draws_pd(vars=param_name)
    index_2_group = {v: f"{param_name}_{k}" for k, v in group_2_index.items()}
    draws_df.columns = (
        draws_df.columns.map(lambda c: c.split("[")[1].rstrip("]"))
        .map(int)
        .map(index_2_group)
    )
    return draws_df

In [8]:
for param in ["learn", "forget", "guess", "slip", "pi_know"]:
    draws_df = extract_draws(fit, param)
    margin_space_c = draws_df[
        [param + suffix for suffix in ["_CSCC", "_CSIC", "_CSNC"]]
    ].mean(axis=1)
    margin_space_i = draws_df[
        [param + suffix for suffix in ["_ISCC", "_ISIC", "_ISNC"]]
    ].mean(axis=1)
    margin_space_n = draws_df[
        [param + suffix for suffix in ["_NSCC", "_NSIC", "_NSNC"]]
    ].mean(axis=1)
    margin_color_c = draws_df[
        [param + suffix for suffix in ["_CSCC", "_ISCC", "_NSCC"]]
    ].mean(axis=1)
    margin_color_i = draws_df[
        [param + suffix for suffix in ["_CSIC", "_ISIC", "_NSIC"]]
    ].mean(axis=1)
    margin_color_n = draws_df[
        [param + suffix for suffix in ["_CSNC", "_ISNC", "_NSNC"]]
    ].mean(axis=1)
    spacing_effect_c = 100 * (margin_space_c - margin_space_n)
    spacing_effect_i = 100 * (margin_space_i - margin_space_n)
    color_effect_c = 100 * (margin_color_c - margin_color_n)
    color_effect_i = 100 * (margin_color_i - margin_color_n)
    print("+" * 20)
    print(param)
    print("mean, median, 0.025 quantile, 0.975 quantile")
    print(
        "spacing congruent:",
        spacing_effect_c.mean(),
        spacing_effect_c.median(),
        spacing_effect_c.quantile(0.025),
        spacing_effect_c.quantile(0.975),
    )

    print(
        "spacing incongruent:",
        spacing_effect_i.mean(),
        spacing_effect_i.median(),
        spacing_effect_i.quantile(0.025),
        spacing_effect_i.quantile(0.975),
    )

    print(
        "color effect congruent:",
        color_effect_c.mean(),
        color_effect_c.median(),
        color_effect_c.quantile(0.025),
        color_effect_c.quantile(0.975),
    )

    print(
        "color effect incongruent:",
        color_effect_i.mean(),
        color_effect_i.median(),
        color_effect_i.quantile(0.025),
        color_effect_i.quantile(0.975),
    )

++++++++++++++++++++
learn
mean, median, 0.025 quantile, 0.975 quantile
spacing congruent: -0.4733699887089583 -0.48772643333333343 -1.5793361850833334 0.7578097866666663
spacing incongruent: 0.17285015666416667 0.16584922833333318 -0.959417406666667 1.3539333881666662
color effect congruent: -0.4236834617160416 -0.42115117500000016 -1.7256493399166666 0.8089452908333328
color effect incongruent: 0.02248453594 0.04537488333333298 -1.266477430583333 1.19348679075
++++++++++++++++++++
forget
mean, median, 0.025 quantile, 0.975 quantile
spacing congruent: -0.046703064120750246 -0.0451910530333333 -0.9589975783714167 0.8757707192068833
spacing incongruent: 0.5525144183470319 0.5467784940333333 -0.4551330904749999 1.5614572489083332
color effect congruent: -0.935415050877982 -0.9228846044544833 -2.0903881766304173 0.20080319816666628
color effect incongruent: -1.1867437743437808 -1.1550575845333335 -2.278043944415833 -0.2568962842612848
++++++++++++++++++++
guess
mean, median, 0.025 quantil

## Mastery Analysis

In [9]:
mastery_max_df = pd.read_csv("output/04_predict/itv_mastery_max.csv")

In [10]:
group_id = data[["user_id", "template_id", "S1_Pretest_Acc"]].drop_duplicates()

In [11]:
group_id["S1_Pretest_Acc"].median()

np.float64(0.4)

In [12]:
mastery_max_df = mastery_max_df.merge(group_id, on=["user_id"])

In [13]:
mastery_max_df_high_pre = mastery_max_df.loc[mastery_max_df["S1_Pretest_Acc"] > 0.4]
mastery_max_df_low_pre = mastery_max_df.loc[mastery_max_df["S1_Pretest_Acc"] <= 0.4]

In [ ]:
for is_max in [True, False]:
    for df, label in zip(
        [mastery_max_df_high_pre, mastery_max_df_low_pre],
        ["High Pretest", "Low Pretest"],
    ):
        if not is_max:
            df["mastery_max"] = df["mastery_max"] > 0.95
        margin_space_c = (
            df.loc[df["template_id"].str.contains("CS")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        margin_space_i = (
            df.loc[df["template_id"].str.contains("IS")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        margin_space_n = (
            df.loc[df["template_id"].str.contains("NS")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        margin_color_c = (
            df.loc[df["template_id"].str.contains("CC")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        margin_color_i = (
            df.loc[df["template_id"].str.contains("IC")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        margin_color_n = (
            df.loc[df["template_id"].str.contains("NC")]
            .groupby("draw__")["mastery_max"]
            .mean()
        )
        spacing_effect_c = 100 * (margin_space_c - margin_space_n)
        spacing_effect_i = 100 * (margin_space_i - margin_space_n)
        color_effect_c = 100 * (margin_color_c - margin_color_n)
        color_effect_i = 100 * (margin_color_i - margin_color_n)
        print("+" * 20)
        print(label)
        if not is_max:
            print("mastery max > 0.95 (i.e. has mastered)")
        else:
            print("mastery max")
        print("mean, median, 0.025 quantile, 0.975 quantile")
        print(
            "spacing congruent:",
            spacing_effect_c.mean(),
            spacing_effect_c.median(),
            spacing_effect_c.quantile(0.025),
            spacing_effect_c.quantile(0.975),
        )

        print(
            "spacing incongruent:",
            spacing_effect_i.mean(),
            spacing_effect_i.median(),
            spacing_effect_i.quantile(0.025),
            spacing_effect_i.quantile(0.975),
        )

        print(
            "color effect congruent:",
            color_effect_c.mean(),
            color_effect_c.median(),
            color_effect_c.quantile(0.025),
            color_effect_c.quantile(0.975),
        )

        print(
            "color effect incongruent:",
            color_effect_i.mean(),
            color_effect_i.median(),
            color_effect_i.quantile(0.025),
            color_effect_i.quantile(0.975),
        )

++++++++++++++++++++
High Pretest
mastery max
mean, median, 0.025 quantile, 0.975 quantile
spacing congruent: 5.200136267852152 5.230862213372828 1.690961931403058 8.608420621067411
spacing incongruent: 3.6351671448341913 3.6526098098580704 0.13817733196750204 7.073630444630945
color effect congruent: 1.4227896217367524 1.4506234680476182 -2.4866688909166608 5.252492827754748
color effect incongruent: 4.823374416353356 4.826319433575515 1.7635165525718457 7.817032391468976
++++++++++++++++++++
Low Pretest
mastery max
mean, median, 0.025 quantile, 0.975 quantile
spacing congruent: -2.391671344560278 -2.7318898690189775 -8.163592622293503 5.027532745255248
spacing incongruent: 2.357693513451893 2.7359781121751023 -3.1463748290013696 7.466940264477884
color effect congruent: 1.5652040256892248 1.92669172932331 -6.286549707602335 8.808479532163743
color effect incongruent: 4.008082706766917 4.511278195488721 -3.7593984962406015 11.278195488721805
++++++++++++++++++++
High Pretest
mastery m

In [115]:
margin_space_c = mastery_max_df.groupby()

TypeError: You have to supply one of 'by' and 'level'

In [ ]:
draws_df = extract_draws(fit, "guess")

In [ ]:
param = "guess"

In [ ]:
draws_df

,guess_NSIC,guess_CSCC,guess_ISNC,guess_ISIC,guess_NSNC,guess_CSIC,guess_CSNC,guess_ISCC,guess_NSCC
0,0.206510,0.310487,0.264219,0.188591,0.282493,0.251463,0.381682,0.253148,0.457601
1,0.213761,0.363276,0.203159,0.258985,0.320208,0.249930,0.386054,0.275479,0.481340
2,0.196102,0.379988,0.307359,0.248645,0.264770,0.222535,0.355485,0.191697,0.452865
3,0.205589,0.365940,0.301040,0.215029,0.265215,0.236462,0.319820,0.199267,0.446737
4,0.193131,0.333577,0.267203,0.280644,0.284599,0.231327,0.410737,0.258151,0.462419
...,...,...,...,...,...,...,...,...,...
7995,0.179037,0.346303,0.272034,0.220138,0.281470,0.245827,0.353328,0.263609,0.428863
7996,0.216026,0.358877,0.263544,0.258234,0.322714,0.248033,0.407118,0.229599,0.476792
7997,0.164186,0.351782,0.270500,0.242062,0.288240,0.232818,0.391012,0.259933,0.447767
7998,0.226282,0.338643,0.276052,0.241752,0.331445,0.231691,0.376402,0.232947,0.470746


In [ ]:
margin_space_c = draws_df[
    [param + suffix for suffix in ["_CSCC", "_CSIC", "_CSNC"]]
].mean(axis=1)
margin_space_i = draws_df[
    [param + suffix for suffix in ["_ISCC", "_ISIC", "_ISNC"]]
].mean(axis=1)
margin_space_n = draws_df[
    [param + suffix for suffix in ["_NSCC", "_NSIC", "_NSNC"]]
].mean(axis=1)

In [ ]:
margin_color_c = draws_df[
    [param + suffix for suffix in ["_CSCC", "_ISCC", "_NSCC"]]
].mean(axis=1)
margin_color_i = draws_df[
    [param + suffix for suffix in ["_CSIC", "_ISIC", "_NSIC"]]
].mean(axis=1)
margin_color_n = draws_df[
    [param + suffix for suffix in ["_CSNC", "_ISNC", "_NSNC"]]
].mean(axis=1)

In [ ]:
spacing_effect_c = margin_space_c - margin_space_n
spacing_effect_i = margin_space_i - margin_space_n
color_effect_c = margin_color_c - margin_color_n
color_effect_i = margin_color_i - margin_color_n

In [ ]:
print("+" * 20)
print(param)
print(
    "spacing-color:",
    spacing_effect_c.mean(),
    spacing_effect_c.median(),
    spacing_effect_c.quantile(0.025),
    spacing_effect_c.quantile(0.975),
)

++++++++++++++++++++
guess
spacing-color: 0.013599445805833334 0.013909771666666654 -0.029707137833333314 0.054664101500000006


In [ ]:
baseline_col = "learn_NSNC"

# Compute posterior draw differences for every learn_* column against learn_NSNC.
diff_draws = learn_draws.subtract(learn_draws[baseline_col], axis=0)
diff_draws = diff_draws.drop(columns=[baseline_col], errors="ignore")

credible_interval = [0.025, 0.975]
diff_summary = (
    diff_draws.agg(["mean", "median"])
    .T.join(diff_draws.quantile(credible_interval).T)
    .rename(columns={0.025: "ci_2.5%", 0.975: "ci_97.5%"})
    .sort_index()
)

diff_summary